# The Recommendation Problem

Companion notebook for the [Recommendation Problem lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/01-the-recommendation-problem).

**The idea in one sentence.** A recommender fills in a giant, mostly-*empty*
user–item matrix — predicting what each user would think of items they've never
seen — and is judged not on rating accuracy but on the **ranking** it puts at the
top of the list.

Two pieces, both from scratch:

- **Collaborative filtering** — "users who agreed in the past will agree again":
  predict a rating as the similarity-weighted average of like-minded users.
- **Ranking metrics** — precision@k, recall@k, and **NDCG@k**, which rewards
  putting relevant items *higher* (a great hit buried at rank 10 barely counts).

We **validate our NDCG against scikit-learn** and cover the sparsity/popularity
gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)

## 1 — A sparse user–item matrix

Rows = users, columns = items, entries = ratings (0 = not yet rated). Most of the matrix is unknown
— that sparsity is the whole challenge.

In [ ]:
R = np.array([
    [5, 4, 0, 0, 1],   # user 0
    [5, 5, 0, 0, 0],   # user 1 (similar to user 0)
    [0, 0, 5, 4, 0],   # user 2
    [1, 0, 4, 5, 0],   # user 3
], dtype=float)
print('user-item ratings (0 = unrated):\n', R)
print('sparsity:', f'{100*(R==0).mean():.0f}% of entries unknown')

## 2 — Memory-based collaborative filtering

Predict user u's rating of item i as the similarity-weighted average of other users' ratings of i.
User similarity is cosine over their rating vectors — the KNN-on-the-matrix idea.

In [ ]:
def cosine_sim(R):
    norms = np.linalg.norm(R, axis=1, keepdims=True)
    Rn = R / np.where(norms == 0, 1, norms)
    return Rn @ Rn.T

def predict(R, S, u, i):
    others = [v for v in range(R.shape[0]) if v != u and R[v, i] > 0]
    if not others:
        return 0.0
    w = np.array([S[u, v] for v in others])
    r = np.array([R[v, i] for v in others])
    return (w @ r) / (np.abs(w).sum() + 1e-9)

S = cosine_sim(R)
print('user-user similarity:\n', S)
# user 0 has not rated item 2 or 3; predict them
print('\npredicted rating user0->item2:', round(predict(R, S, 0, 2), 2))
print('predicted rating user0->item3:', round(predict(R, S, 0, 3), 2))
print('user 0 is most similar to user 1 (sim={:.2f}) who also dislikes items 2,3'.format(S[0,1]))

## 3 — Ranking metrics

A recommender returns a ranked list. precision@k and recall@k ignore order within the top-k; NDCG@k
rewards placing relevant items *higher* via a logarithmic position discount.

In [ ]:
def precision_at_k(ranked, relevant, k):
    topk = ranked[:k]
    return sum(i in relevant for i in topk) / k

def recall_at_k(ranked, relevant, k):
    topk = ranked[:k]
    return sum(i in relevant for i in topk) / len(relevant)

def ndcg_at_k(ranked, relevant, k):
    dcg = sum((1.0 if item in relevant else 0.0) / np.log2(rank + 2)
              for rank, item in enumerate(ranked[:k]))
    ideal = sum(1.0 / np.log2(rank + 2) for rank in range(min(len(relevant), k)))
    return dcg / ideal if ideal > 0 else 0.0

relevant = {1, 3, 4}
good = [1, 3, 0, 4, 2]      # relevant items near the top
bad  = [0, 2, 1, 3, 4]      # same items present, but lower
for name, r in [('good ranking', good), ('bad ranking', bad)]:
    print(f'{name}: P@3={precision_at_k(r,relevant,3):.2f}  R@3={recall_at_k(r,relevant,3):.2f}  NDCG@5={ndcg_at_k(r,relevant,5):.3f}')
print('\nSame items, but NDCG rewards the ranking that puts relevant items higher.')

### Validate NDCG against scikit-learn

NDCG is subtle (the log discount, the ideal-DCG normaliser), so we check our
implementation against `sklearn.metrics.ndcg_score` on the same ranking.

In [ ]:
from sklearn.metrics import ndcg_score

n = 5
y_true = np.zeros(n)
for it in relevant:
    y_true[it] = 1.0
# turn our ranked list into per-item scores (higher = ranked earlier)
y_score = np.zeros(n)
for rank, it in enumerate(good):
    y_score[it] = n - rank
sk = ndcg_score([y_true], [y_score], k=5)
ours = ndcg_at_k(good, relevant, 5)
print(f'our NDCG@5     : {ours:.6f}')
print(f'sklearn NDCG@5 : {sk:.6f}')
assert np.isclose(ours, sk), 'our NDCG must match sklearn'
print('\n✅ NDCG matches scikit-learn')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **sparsity** | most entries are unknown; naive averages are unreliable with few co-ratings |
| **cold start** | a brand-new user/item has no history → CF can't score it (needs content features) |
| **popularity bias** | recommending only popular items is safe but drowns the long tail |
| **precision@k ignores order** | two lists with the same top-k set score equally; use NDCG to reward ranking |
| **cosine on raw ratings** | mixes "rated highly" with "rated at all"; mean-centering per user helps |

Demo: a personalized CF ranking beats the popularity baseline for a niche user.

In [ ]:
# popularity-baseline vs personalization: recommend the globally most-rated items to everyone
pop = (R > 0).sum(0)                      # how many users rated each item
pop_ranking = list(np.argsort(-pop))
print('global popularity ranking:', pop_ranking)
# for user 2 (likes items 2,3), the personalized CF prediction should rank 2,3 above the
# globally-popular items 0,1 that user 2 dislikes:
S2 = cosine_sim(R)
personal = sorted(range(R.shape[1]), key=lambda i: -predict(R, S2, 2, i))
print('personalized ranking for user 2:', personal)
assert personal[0] in (2, 3), 'personalization should beat popularity for a niche user'
print('\nPopularity is a strong but non-personalized baseline; CF tailors to the user.')

## ✏️ Your turn

**Exercise.** Implement `dcg_at_k(ranked, relevant, k)` (the un-normalized discounted cumulative
gain) and `ndcg(ranked, relevant, k)` (normalize by the ideal DCG so the score lands in [0, 1]).
Relevant item at position `rank` (0-indexed) contributes `1 / log2(rank + 2)`.

In [ ]:
def dcg_at_k(ranked, relevant, k):
    # TODO(you): sum 1/log2(rank+2) over the top-k positions whose item is relevant
    return ...

def ndcg(ranked, relevant, k):
    # TODO(you): dcg_at_k divided by the ideal DCG (all relevant items packed at the top)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
rel = {1, 3, 4}
# DCG of a single relevant item at rank 0 is 1/log2(2) = 1.0
assert abs(dcg_at_k([1, 0, 2], rel, 3) - 1.0) < 1e-9
# a perfect ranking (all relevant first) has NDCG 1.0
assert abs(ndcg([1, 3, 4, 0, 2], rel, 5) - 1.0) < 1e-9
# matches the reference implementation from section 3
assert abs(ndcg([1, 3, 0, 4, 2], rel, 5) - ndcg_at_k([1, 3, 0, 4, 2], rel, 5)) < 1e-9
assert ndcg([1, 3, 4, 0, 2], rel, 5) > ndcg([0, 2, 1, 3, 4], rel, 5)
print('✓ DCG and NDCG are correct')

<details>
<summary>Solution</summary>

```python
def dcg_at_k(ranked, relevant, k):
    return sum((1.0 if item in relevant else 0.0) / np.log2(rank + 2)
               for rank, item in enumerate(ranked[:k]))

def ndcg(ranked, relevant, k):
    ideal = sum(1.0 / np.log2(r + 2) for r in range(min(len(relevant), k)))
    return dcg_at_k(ranked, relevant, k) / ideal if ideal > 0 else 0.0
```

The log discount makes rank 1 worth far more than rank 10, so NDCG captures what users feel: a great
recommendation buried at the bottom of the list barely counts.

</details>

## Key takeaways

- **Recommendation = matrix completion + ranking.** Most of the user–item matrix
  is unknown; the job is to predict and then *order* the top-k well.
- **Collaborative filtering** predicts from similar users' ratings — no item
  content needed, just the co-rating pattern.
- **NDCG rewards position**, not just presence — a relevant item at rank 1 is
  worth far more than at rank 10. We matched scikit-learn's implementation.
- **Popularity is a strong baseline** but ignores the individual; personalization
  wins for niche tastes — at the cost of the **cold-start** and **sparsity**
  problems the rest of this course tackles.